In [3]:
# Example_numerics.ipynb
#
# Producing numerical estimates for the product of rho(p) when l,m,n are pairwise coprime
#
# Authors: Christopher Keyes, Andrew Kobin
# Updated 25 February 2026

load('gfedensity.sage')

# prod_rho_numerics
#     computes upper and lower bounds for the product of rho(p) when l,m,n pairwise coprime
# INPUTS:
#     * l,m,n: pairwise coprime exponents
#     * M: positive integer (higher improves precision)
# OUTPUTS:
#     * ub_sofar: upper bound for product of rho(p)
#     * lb_sofar: lower bound for product of rho(p)
def prod_rho_numerics(l,m,n,M):
    assert gcd(l,m) == 1 and gcd(l,n) == 1 and gcd(m,n) == 1
    
    # get probabilities
    r, rA, rB, rC, rAB, rAC, rBC = get_probs(l,m,n)
    rho = r(gcd_pmin1_l_m = 1, gcd_pmin1_l_n = 1, gcd_pmin1_m_n = 1) # specialize gcd variables

    # initialize upper andlower bounds
    ub_sofar = 1/zeta(3)
    lb_sofar = 1/zeta(2)^4 
    
    # get upper and lower bounds
    for p0 in [p0 for p0 in [1 .. M] if p0.is_prime()]:
        ub_sofar = ub_sofar * (1 - 1/p0^3)^-1 * rho(p=p0)
        lb_sofar = lb_sofar * (1 - 1/p0^2)^-4 * rho(p=p0)
        
    return ub_sofar, lb_sofar

# prod_rho_numerics_numberfield
#     version of prod_rho_numerics for number fields. Uses prime ideals of norm
#     at most M. Must have l,m,n pairwise coprime.
# INPUTS:
#     * l,m,n: pairwise coprime exponents
#     * M: positive integer (higher improves precision)
#     * K: a number field
# OUTPUTS:
#     * ub_sofar: upper bound for product of rho(p)
#     * lb_sofar: lower bound for product of rho(p)
def prod_rho_numerics_numberfield(l,m,n,M,K):
    assert gcd(l,m) == 1 and gcd(l,n) == 1 and gcd(m,n) == 1
    zetaK = K.zeta_function()
    
    # get probabilities
    r, rA, rB, rC, rAB, rAC, rBC = get_probs(l,m,n)
    rho = r(gcd_pmin1_l_m = 1, gcd_pmin1_l_n = 1, gcd_pmin1_m_n = 1) # specialize gcd variables

    # initialize upper andlower bounds
    ub_sofar = 1/zetaK(3)
    lb_sofar = 1/zetaK(2)^4 
    
    # get upper and lower bounds
    for p0 in [p0 for p0 in K.primes_of_bounded_norm(M)]:
        q = p0.norm()
        ub_sofar = ub_sofar * (1 - 1/q^3)^-1 * rho(p=q)
        lb_sofar = lb_sofar * (1 - 1/q^2)^-4 * rho(p=q)
        
    return ub_sofar, lb_sofar

In [4]:
# get numerics in an example
(l,m,n) = (2,3,5)
M = 10000
ub, lb = prod_rho_numerics(l,m,n,M)

# printing
print("(l,m,n) =", (l,m,n))
print("M =", M)
print(float(lb), "<= prod_p(rho(p)) <=", float(ub))

(l,m,n) = (2, 3, 5)
M = 10000
0.7823370435594876 <= prod_p(rho(p)) <= 0.782367760508551


In [0]:
# get numerics in an example
(l,m,n) = (2,3,7)
M = 10000
ub, lb = prod_rho_numerics(l,m,n,M)

# printing
print("(l,m,n) =", (l,m,n))
print("M =", M)
print(float(lb), "<= prod_p(rho(p)) <=", float(ub))

In [0]:
# experimenting with (2,3,5) example over a number field
(l,m,n) = (2,3,5)
M = 1000
Q = Rationals()
R.<x> = PolynomialRing(Q)

# Let's compute the product of rhoK(p) for some handpicked quadratic and cubic fields: 
#     * Q(sqrt(-479)): the primes 2, 3, 5, 7, 11 are all split
#     * Q(sqrt(-403)): 2, 3, 5, and 7 are inert
#     * Q[x]/(x^3 - 49x + 72): 2 and 3 are split
#     * Q[x]/(x^3 - 49x + 31): 2, 3, 5, and 7 are all inert.
# The presence of many/few small primes greatly influences the product of densities.

# the quadratic fields
dlist = [-479, -403]
for d in dlist:
    K.<a> = Q.extension(x^2 - d)
    smallprimes = K.primes_of_bounded_norm(10)
    ub, lb = prod_rho_numerics_numberfield(l,m,n,M,K)
    print(K)
    print(float(lb), "<= prod_p(rhoK(p)) <=", float(ub))
    print("norms of primes up to 10:", [q.norm() for q in K.primes_of_bounded_norm(10)])
    print()

# the cubic fields
bclist = [(-49,72), (-49,31)]
for (b,c) in bclist:
    f = x^3 + b*x + c
    K.<a> = NumberField(f)
    ub, lb = prod_rho_numerics_numberfield(l,m,n,M,K)
    print(K)
    print(float(lb), "<= prod_p(rhoK(p)) <=", float(ub))
    print("norms of primes up to 10:", [q.norm() for q in K.primes_of_bounded_norm(10)])
    print()